# Detector perro vs no-perro — **TensorFlow** (CNN desde cero)

Ejecuta en orden (o **Run All**): dependencias → dataset → entrenamiento → resultados.


In [ ]:
# 1.0) Preparar CUDA para TensorFlow antes de importar el framework
import os
from pathlib import Path

venv_root = Path('/mnt/c/Users/gotts/AI-Frameworks/venv')
nvidia_root = venv_root / 'lib/python3.12/site-packages/nvidia'
if nvidia_root.exists():
    lib_paths = [str(path) for path in nvidia_root.glob('*/lib')]
    current_ld = os.environ.get('LD_LIBRARY_PATH', '')
    merged = ':'.join(lib_paths + ['/usr/lib/wsl/lib'] + ([current_ld] if current_ld else []))
    os.environ['LD_LIBRARY_PATH'] = merged
    print('LD_LIBRARY_PATH configurado para TensorFlow GPU')
else:
    print('No se encontró', nvidia_root)


In [1]:
# 1) Dependencias (si falta algo, descomenta el pip)
# %pip install -r requirements.txt
import importlib
for pkg in ['tensorflow', 'numpy', 'matplotlib', 'PIL', 'tqdm']:
    mod = importlib.import_module(pkg)
    print(f'{pkg}: {getattr(mod, "__version__", "ok")}')

I0000 00:00:1783264618.155621    1250 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1783264618.301332    1250 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1783264620.082339    1250 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1783264690.517055    1250 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

tensorflow: 2.21.0
numpy: 2.5.0
matplotlib: 3.11.0
PIL: 12.3.0
tqdm: 4.68.3


In [ ]:
# 2) Dataset (idempotente: si ya está, no re-descarga)
import dataset
dataset.build()

In [7]:
# 3) Entrenamiento de la CNN desde cero (TensorFlow)
import train_tf
results = train_tf.run(epochs=30)

TensorFlow: 2.21.0 | GPU: no (CPU)
Found 7998 files belonging to 2 classes.
Using 6399 files for training.
Found 7998 files belonging to 2 classes.
Using 1599 files for validation.
Clases: ['dog', 'not_dog'] | class_weight: {0: 1.0, 1: 1.0}


Model: "dog_detector"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip_1 (RandomFlip)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_1               │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_1 (RandomZoom)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast_1               │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 224, 224, 32)   │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_6 (ReLU)                  │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_7 (ReLU)                  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_8 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 56, 56, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_9 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 28, 28, 128)    │             

 Total params: 1,194,721 (4.56 MB)

 Trainable params: 1,192,993 (4.55 MB)

 Non-trainable params: 1,728 (6.75 KB)

Epoch 1/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 441s 2s/step - accuracy: 0.5760 - auc: 0.6040 - loss: 0.6951 - val_accuracy: 0.5078 - val_auc: 0.5889 - val_loss: 0.7530 - learning_rate: 0.0010
Epoch 2/30
200/200 ━━━━━━━━━━━━━━━━━━━━ 421s 2s/step - accuracy: 0.6148 - auc: 0.6575 - loss: 0.6529 - val_accuracy: 0.5097 - val_auc: 0.6171 - val_loss: 0.8188 - learning_rate: 0.0010
Epoch 3/30
 87/200 ━━━━━━━━━━━━━━━━━━━━ 3:56 2s/step - accuracy: 0.6333 - auc: 0.6788 - loss: 0.6414

KeyboardInterrupt: 

In [5]:
# 4) Resultados
print('Clases:', results['class_names'])
print('Matriz de confusión:'); print(results['confusion_matrix'])
print(f"Accuracy: {results['accuracy']:.4f}")
from IPython.display import Image
Image(filename='artifacts/dog_detector_tf_history.png')

NameError: name 'results' is not defined

In [3]:
from pathlib import Path
import matplotlib.pyplot as plt

confusion_matrix = results['confusion_matrix']
class_names = results['class_names']
confusion_matrix_path = Path.cwd() / 'artifacts' / 'dog_detector_tf_confusion_matrix.png'
confusion_matrix_path.parent.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(confusion_matrix, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de confusión')
for i in range(confusion_matrix.shape[0]):
    for j in range(confusion_matrix.shape[1]):
        ax.text(j, i, str(confusion_matrix[i, j]), ha='center', va='center', color='black')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(confusion_matrix_path, dpi=120)
plt.show()
plt.close(fig)
print('Matriz de confusión guardada en', confusion_matrix_path)

NameError: name 'results' is not defined

In [4]:
# 5) Inferencia sobre una imagen nueva
from pathlib import Path
import tensorflow as tf
import train_tf

inference_model = tf.keras.models.load_model(train_tf.MODEL_PATH)

def predict_image(image_path, class_names=None):
    image_path = Path(image_path)
    image = tf.keras.utils.load_img(image_path, target_size=train_tf.CONFIG.image_size)
    image_array = tf.keras.utils.img_to_array(image)
    image_array = tf.expand_dims(image_array, axis=0) / 255.0

    probability = float(inference_model.predict(image_array, verbose=0).ravel()[0])
    label_index = int(probability >= 0.5)
    if class_names is None:
        class_names = results['class_names'] if 'results' in globals() else ['not_dog', 'dog']
    label = class_names[label_index]
    print(f'Imagen: {image_path}')
    print(f'Probabilidad de dog: {probability:.4f}')
    print(f'Resultado: {label}')
    return {'image_path': str(image_path), 'prob_dog': probability, 'label': label}

# Cambia esta ruta por tu foto
sample_image_path = 'ruta/a/tu/foto.jpg'
# predict_image(sample_image_path)